# Etapa 6: Feature Engineering

---

In [ ]:
import sys
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression

In [33]:
# Ruta raíz del proyecto (cwd = donde se encuentra el notebook; .parent = ruta padre, eso da la ruta raíz)
PROJECT_ROOT = Path.cwd().parent

RAW_PATH = PROJECT_ROOT / "data" / "processed" / "bank_marketing.csv"

df = pd.read_csv(RAW_PATH)

In [34]:
ROOT_DIR = Path.cwd().parent
sys.path.append(str(ROOT_DIR))

In [ ]:
from src.features.feature_engineering import build_preprocessor

preprocessor = build_preprocessor()
print(preprocessor)

ColumnTransformer(transformers=[('numeric', 'passthrough',
                                 ['age', 'balance', 'day', 'campaign', 'pdays',
                                  'previous']),
                                ('categorical',
                                 OneHotEncoder(handle_unknown='ignore'),
                                 ['job', 'marital', 'education', 'default',
                                  'housing', 'loan', 'contact', 'month',
                                  'poutcome'])])


In [41]:
# Separación de las variables predictoras (X) de la variable objetivo (y)
# Además, se elimina 'duration' porque podría introducir data leakage en el modelo
X = df.drop(columns=["y", "duration"])
y = df["y"]

print("Dimensiones de X:", X.shape)
print("Dimensiones de y:", y.shape)

# Verificar las variables que serán utilizadas como predictoras
print("Variables predictoras:")
print(X.columns.tolist())

Dimensiones de X: (45211, 15)
Dimensiones de y: (45211,)
Variables predictoras:
['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day', 'month', 'campaign', 'pdays', 'previous', 'poutcome']


In [ ]:
from sklearn.model_selection import train_test_split

# Dividimos los datos en entrenamiento (80%) y prueba (20%).
# stratify mantiene aproximadamente la misma proporción de clases yes/no en ambos conjuntos.
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [38]:
# Comparamos la distribución de la variable objetivo entre entrenamiento y prueba
# para comprobar que el desbalance de clases se mantiene aproximadamente igual.

print("Distribución en entrenamiento:")
print(y_train.value_counts(normalize=True).round(3))

print("\nDistribución en prueba:")
print(y_test.value_counts(normalize=True).round(3))

Distribución en entrenamiento:
y
no     0.883
yes    0.117
Name: proportion, dtype: float64

Distribución en prueba:
y
no     0.883
yes    0.117
Name: proportion, dtype: float64


In [ ]:
# Construimos los preprocesador definido en feature_engineering.py

# Modelos de árboles con incluir_escalado=False
preprocessor_trees = build_preprocessor(incluir_escalado=False)

# Modelos de KNN y RL con incluir_escalado=True
preprocessor_knn_rl = build_preprocessor(incluir_escalado=True)

In [ ]:
# Creamos el pipeline de Decision Tree.
# Primero transforma las variables y después aplica el modelo.
decision_tree_pipeline = Pipeline([
    ("preprocessor", preprocessor_trees),
    ("model", DecisionTreeClassifier(
        random_state=42
    ))
])

print("Pipeline de Decision Tree creado correctamente.")

Pipeline de Decision Tree creado correctamente.


In [ ]:
# Creamos el pipeline de Random Forest.
# Primero transforma las variables y después aplica el modelo.
random_forest_pipeline = Pipeline([
    ("preprocessor", preprocessor_trees),
    ("model", RandomForestClassifier(
        random_state=42
    ))
])

print("Pipeline de Random Forest creado correctamente.")

Pipeline de Random Forest creado correctamente.


In [ ]:
k=2 # TODO

# Creamos el pipeline de KNN
# Primero transforma las variables, aplica escalado y después corre el modelo.
knn_pipeline = Pipeline([
    ("preprocessor", preprocessor_knn_rl),
    ("model", KNeighborsClassifier(
        n_neighbors=k, 
        random_state=42
    ))
])

print("Pipeline de KNN creado correctamente.")

In [ ]:
# Creamos el pipeline de Regresión Logística
# Primero transforma las variables, aplica escalado y después corre el modelo.
rl_pipeline = Pipeline([
    ("preprocessor", preprocessor_knn_rl),
    ("model", KNeighborsClassifier(
        random_state=42
    ))
])

print("Pipeline de Regresión Logística creado correctamente.")